# 04 - Continuous OptBinning for Amount + Explained Variance

This notebook applies continuous optimal binning against deal amount and reports:
- binning-based IV-like score from continuous tables
- explained variance from bin-mean predictions
- ANOVA F-statistic over bin groups

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import f_oneway
from sklearn.metrics import explained_variance_score
from optbinning import ContinuousOptimalBinning

pd.set_option('display.max_columns', 200)

In [3]:
df_load = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df_load.shape)
df_load.head(2)

shape: (53073, 37)


,Opportunity Number,Supplies Group,Supplies Subgroup,Region,Route To Market,Elapsed Days In Sales Stage,Opportunity Result,Sales Stage Change Count,Total Days Identified Through Closing,Total Days Identified Through Qualified,Opportunity Amount USD,Client Size By Revenue (USD),Client Size By Employee Count,Revenue From Client Past Two Years (USD),Competitor Type,Ratio Days Identified To Total Days,Ratio Days Validated To Total Days,Ratio Days Qualified To Total Days,Deal Size Category (USD),total_days_zero,Opportunity Result Bool,opportunity_amount_weirdness,row_position,flag_0_days,flag_ratio_problem,flag_zero_opportunity_amount,flag_outlier_opportunity_amount,flag_outlier_total_days,flag_totally_repeated_row,flag_partially_repeated_row,partial_repeat_is_latest_id_appearance,flag_only_identified,flag_weirdness_over_75th_pct,problem_tags,problem_count,amount_qbin,stratify_key
0,9406532,Car Accessories,Batteries & Accessories,Midwest,Reseller,25,Loss,1,0,0,92024,100K or less,1K or less,0 (No business),Unknown,0.0,0.0,0.0,30K to 40K,True,False,4.855807,39993,True,True,False,False,False,False,False,False,False,False,"[0_days, ratio_problem]",2,"(65789.0, 100000.0]","0__(65789.0, 100000.0]"
1,9953365,Car Accessories,Garage & Car Care,Pacific,Reseller,3,Won,3,2,2,1550,100K or less,1K or less,0 (No business),NaN,0.0,1.0,0.0,10K or less,False,True,6.842306,45225,False,False,False,False,False,False,False,False,False,False,[],0,"(0.999, 6500.0]","1__(0.999, 6500.0]"


In [4]:
columns_to_use = ['Supplies Group', 'Supplies Subgroup', 'Region',
       'Route To Market', 'Elapsed Days In Sales Stage', 'Opportunity Result',
       'Sales Stage Change Count', 'Total Days Identified Through Closing',
       'Total Days Identified Through Qualified', 'Opportunity Amount USD',
       'Client Size By Revenue (USD)', 'Client Size By Employee Count',
       'Revenue From Client Past Two Years (USD)', 'Competitor Type',
       'Ratio Days Identified To Total Days',
       'Ratio Days Validated To Total Days',
       'Ratio Days Qualified To Total Days', 'Deal Size Category (USD)', "Opportunity Result Bool"]

df = df_load[columns_to_use].copy()

In [5]:
df['target_amount'] = df['Opportunity Amount USD'].astype(float)
df['target_log_amount'] = np.log1p(df['target_amount'])
print('shape:', df.shape)
print(df[['target_amount', 'target_log_amount']].describe())

shape: (53073, 21)
        target_amount  target_log_amount
count    53073.000000       53073.000000
mean     94148.126901          10.593797
std     134143.645557           1.563316
min          1.000000           0.693147
25%      17000.000000           9.741027
50%      50000.000000          10.819798
75%     110000.000000          11.608245
max    1000000.000000          13.815512


## Fit continuous optimal binning and compute explained variance

In [6]:
excluded = {'Opportunity Amount USD', 'target_amount', 'target_log_amount', 'Opportunity Number'}
features = [c for c in df.columns if c not in excluded]

rows = []
failed = []
all_variable_tables = []
y = df['target_amount']
for col in features:
    x = df[col]
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'

    try:
        cob = ContinuousOptimalBinning(name=col, dtype=dtype)
        cob.fit(x, y)

        bt = cob.binning_table.build()
        bt_copy = bt.copy()
        bt_copy.insert(0, 'variable', col)
        bt_copy.insert(1, 'dtype', dtype)
        bt_copy.insert(2, 'status', cob.status)
        all_variable_tables.append(bt_copy)
        iv_total = float(bt.iloc[-1]['IV']) if 'IV' in bt.columns else np.nan

        idx = cob.transform(x, metric='indices')
        work = pd.DataFrame({'idx': idx, 'y': y})
        bin_mean = work.groupby('idx', dropna=False)['y'].mean()
        yhat = work['idx'].map(bin_mean).astype(float).to_numpy()
        ev = float(explained_variance_score(y, yhat))

        groups = [g['y'].values for _, g in work.groupby('idx') if len(g) > 1]
        if len(groups) >= 2:
            f_stat = float(f_oneway(*groups).statistic)
        else:
            f_stat = np.nan

        rows.append({
            'variable': col,
            'dtype': dtype,
            'status': cob.status,
            'iv_continuous': iv_total,
            'explained_variance': ev,
            'anova_f': f_stat
        })
    except Exception as e:
        failed.append({'variable': col, 'error': str(e)[:200]})

amount_bin_df = pd.DataFrame(rows)
if not amount_bin_df.empty:
    amount_bin_df = amount_bin_df.sort_values('explained_variance', ascending=False).reset_index(drop=True)
failed_df = pd.DataFrame(failed)
all_variable_info_df = pd.concat(all_variable_tables, ignore_index=True) if all_variable_tables else pd.DataFrame()

def normalize_bin_value(value):
    # Fix optbinning ArrowStringArray/list-like bins into readable names
    if isinstance(value, str) or value is None:
        return value
    if hasattr(value, '__iter__'):
        try:
            vals = [str(v).strip() for v in list(value)]
            vals = [v for v in vals if v]
            if vals:
                return ' | '.join(vals)
        except TypeError:
            pass
    return value

if not all_variable_info_df.empty and 'Bin' in all_variable_info_df.columns:
    all_variable_info_df['Bin'] = all_variable_info_df['Bin'].apply(normalize_bin_value)
print('processed:', len(amount_bin_df), 'failed:', len(failed_df))
print('all variable rows:', len(all_variable_info_df))
amount_bin_df.head(20)

processed: 18 failed: 0
all variable rows: 133


,variable,dtype,status,iv_continuous,explained_variance,anova_f
0,Deal Size Category (USD),categorical,OPTIMAL,84903.991613,0.784614,38662.833499
1,Route To Market,categorical,OPTIMAL,36306.354119,0.073289,4197.132914
2,Supplies Subgroup,categorical,OPTIMAL,24601.259132,0.046070,427.139112
3,Competitor Type,categorical,OPTIMAL,14733.617668,0.023048,1252.057942
4,Ratio Days Identified To Total Days,numerical,OPTIMAL,14493.834615,0.013076,234.374686
5,Revenue From Client Past Two Years (USD),categorical,OPTIMAL,7384.902216,0.012834,689.979090
6,Ratio Days Validated To Total Days,numerical,OPTIMAL,12135.739315,0.012754,137.116224
7,Client Size By Revenue (USD),categorical,OPTIMAL,7276.014821,0.010921,195.322373
8,Ratio Days Qualified To Total Days,numerical,OPTIMAL,10692.534128,0.010704,143.544452
9,Client Size By Employee Count,categorical,OPTIMAL,5925.152081,0.008270,110.635457


In [7]:
output_dir = Path('../../data/bivariate_info')
excel_path = output_dir / 'amount_optbinning_report.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    amount_bin_df.to_excel(writer, sheet_name='variable_summaries', index=False)
    all_variable_info_df.to_excel(writer, sheet_name='all_variable_info', index=False)

print('saved excel:', excel_path)
print('summary rows:', len(amount_bin_df))
print('all-variable rows:', len(all_variable_info_df))
print('saved failures:', (not failed_df.empty))

saved excel: ../../data/bivariate_info/amount_optbinning_report.xlsx
summary rows: 18
all-variable rows: 133
saved failures: False


## Top variable continuous binning table

In [9]:
if amount_bin_df.empty:
    print('No successful continuous optbinning fits to display.')
else:
    top_var = amount_bin_df.loc[0, 'variable']
    dtype = 'numerical' if pd.api.types.is_numeric_dtype(df[top_var]) else 'categorical'
    cob = ContinuousOptimalBinning(name=top_var, dtype=dtype)
    cob.fit(df[top_var], df['target_amount'])
    print('Top variable:', top_var)
    display(cob.binning_table.build())

Top variable: Deal Size Category (USD)


,Bin,Count,Count (%),Sum,Std,Mean,Min,Max,Zeros count,WoE,IV
0,[10K or less],6998,0.131856,3.010425e+07,2761.34553,4301.836525,1.0,9999.0,0,-89846.290376,11846.783488
1,[10K to 20K],10557,0.198915,1.653092e+08,4406.00454,15658.726532,10000.0,24999.0,0,-78489.400369,15612.695715
2,[20K to 30K],8395,0.158178,2.830879e+08,6850.287313,33721.006909,25000.0,49999.0,0,-60427.119992,9558.262626
3,[30K to 40K],9459,0.178226,6.021350e+08,14399.661073,63657.360292,50000.0,99999.0,0,-30490.766609,5434.253978
4,[40K to 50K],12680,0.238916,1.773478e+09,40491.28389,139864.220268,100000.0,249999.0,0,45716.093367,10922.315752
5,"[50K to 60K, More than 60K]",4984,0.093908,2.142609e+09,191246.812997,429897.466693,250000.0,1000000.0,0,335749.339793,31529.680054
6,Special,0,0.000000,0.000000e+00,NaN,0.000000,NaN,NaN,0,-94148.126901,0.000000
7,Missing,0,0.000000,0.000000e+00,NaN,0.000000,NaN,NaN,0,-94148.126901,0.000000
Totals,,53073,1.000000,4.996724e+09,,94148.126901,1.0,1000000.0,0,829015.264307,84903.991613
